## Massachusetts House Roll Call PDF Collection

This web scraper designed to systematically download roll call voting records from the Massachusetts House of Representatives website (2001-2025). The scraper adapts between two API methods: consolidated PDFs for recent sessions (2011+) and individual roll call downloads for historical data (2001-2010).
- Session Coverage: 182-194(2001-2025)

### 1.Libraries, Configuration and Constants

In [2]:
"""--- Test No.1 Roll Call in 183 session---
import requests

url = "https://malegislature.gov/Journal/DownloadRollCall"
data = {
    'GeneralCourtNumber': '183',
    'Branch': 'House',
    'SelectedMonth': '', # 1/1/2004 12:00:00 AM
    'RollCallNumber': '1'
}

resp = requests.post(url, data=data)
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")

if 'pdf' in resp.headers.get('Content-Type', ''):
    with open('test.pdf', 'wb') as f:
        f.write(resp.content)
    print("✅ PDF downloaded successfully!")
else:
    print("❌ Not a PDF response")
    print(resp.text[:500]) 
    """

Status: 200
Content-Type: application/pdf
✅ PDF downloaded successfully!


Imports standard Python libraries for file I/O, regex matching, PDF processing, and data handling.

In [3]:
import requests
import os
import time
import random
from bs4 import BeautifulSoup
from urllib.parse import urljoin

Define configuration and constants

In [4]:
BASE = "https://malegislature.gov"
JOURNAL_BASE = f"{BASE}/Journal/House"
CHUNK_SIZE = 2048
DELAY_MIN = 0.3
DELAY_MAX = 0.8
SAVE_ROOT = "./HJRollCalls"
REQUEST_TIMEOUT = 20
MAX_RETRIES = 2
MAX_CONSECUTIVE_FAILS = 5  # 连续失败5次后停止

In [5]:
SESSION_YEARS = {
    182: [2001, 2002],
    183: [2003, 2004],
    184: [2005, 2006],
    185: [2007, 2008],
    186: [2009, 2010],
    187: [2011, 2012],
    188: [2013, 2014],
    189: [2015, 2016],
    190: [2017, 2018],
    191: [2019, 2020],
    192: [2021, 2022],
    193: [2023, 2024],
    194: [2025],
}

HTTP Utilities

In [ ]:
def safe_get(url, stream=False, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES):
    """Execute HTTP GET request with automatic retry logic"""
    for attempt in range(retries + 1):
        try:
            resp = requests.get(url, stream=stream, timeout=timeout)
            return resp
        except Exception as e:
            if attempt < retries:
                time.sleep(0.5 + attempt * 0.5)
                continue
            print(f"    Request failed after {retries} retries: {e}")
            return None

In [ ]:
def safe_post(url, data, stream=False, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES):
    """Execute HTTP POST request with automatic retry logic"""
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, data=data, stream=stream, timeout=timeout)
            return resp
        except Exception as e:
            if attempt < retries:
                time.sleep(0.5 + attempt * 0.5)
                continue
            print(f"    Request failed after {retries} retries: {e}")
            return None



### 3. File Operations

In [ ]:
def is_pdf_response(resp):
    """Validate response is a valid PDF by checking Content-Type header"""
    if not resp or resp.status_code != 200:
        return False
    ct = resp.headers.get("Content-Type", "")
    return "pdf" in ct.lower()


In [ ]:
def ensure_dir(path):
    """Create directory structure if it doesn't exist"""
    os.makedirs(path, exist_ok=True)


In [ ]:
def save_pdf(resp, save_path):
    """Stream and save PDF content to disk in chunks"""
    ensure_dir(os.path.dirname(save_path))
    with open(save_path, "wb") as f:
        if hasattr(resp, "iter_content"):
            for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
        else:
            f.write(resp.content)

### 4. Legacy Method (2001-2010)

In [ ]:
def download_by_rollcall_number(session, use_delay=True):
    """
    Download roll calls by sequential POST requests to /Journal/DownloadRollCall
    Iterates through roll call numbers until MAX_CONSECUTIVE_FAILS is reached.
    Form Data: GeneralCourtNumber, Branch, RollCallNumber
    """
    saved_files = []
  
    target_dir = os.path.join(SAVE_ROOT, f"session_{session}")  # 按Session分文件夹
    ensure_dir(target_dir)

    years = SESSION_YEARS.get(session, [])
    year_range = '-'.join(map(str, years)) if years else 'Unknown'
    
    print(f"  Downloading Roll Calls for Session {session} ({year_range})...")
    
    rollcall_num = 1
    consecutive_fails = 0
    
    download_url = f"{BASE}/Journal/DownloadRollCall"
    
    while consecutive_fails < MAX_CONSECUTIVE_FAILS:
        if use_delay:
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        
        form_data = {
            'GeneralCourtNumber': str(session),
            'Branch': 'House',
            'SelectedMonth': '',
            'RollCallNumber': str(rollcall_num)
        }
        
        resp = safe_post(download_url, data=form_data, stream=True)
        
        if resp and is_pdf_response(resp):
            filename = f"session{session}_rollcall_{rollcall_num:04d}.pdf"
            save_path = os.path.join(target_dir, filename)
            
            if os.path.exists(save_path):
                print(f"    #{rollcall_num} already exists")
            else:
                print(f"    #{rollcall_num} downloading...")
                save_pdf(resp, save_path)
            
            saved_files.append(save_path)
            consecutive_fails = 0 
        else:
            consecutive_fails += 1
            if consecutive_fails == 1:
                print(f"    #{rollcall_num} not found")
        
        rollcall_num += 1
    
    print(f"  Stopped at Roll Call #{rollcall_num-1} (consecutive fails: {consecutive_fails})")
    return saved_files

### 5. Modern Method (2011+)

In [ ]:
def extract_pdf_links(html_text, base_url):
    """Parse HTML page and extract all PDF links using BeautifulSoup"""
    soup = BeautifulSoup(html_text, "html.parser")
    links = set()
    
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.lower().endswith(".pdf"):
            links.add(urljoin(base_url, href))
    
    return list(links)

In [ ]:
def download_from_index(session, year, use_delay=True):
    """
    Download from consolidated index pages at /Journal/House/{session}/{year}/RollCalls
    """
    saved_files = []
    target_dir = os.path.join(SAVE_ROOT, str(year))
    ensure_dir(target_dir)
    
    base_url = f"{JOURNAL_BASE}/{session}/{year}/RollCalls"
    
    if use_delay:
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
    
    print(f"  Trying index page: {base_url}")
    resp = safe_get(base_url, stream=True)
    
    if not resp:
        return saved_files
    
    if is_pdf_response(resp):
        filename = f"{session}_{year}_RollCalls.pdf"
        save_path = os.path.join(target_dir, filename)
        
        if not os.path.exists(save_path):
            print(f"  Found consolidated PDF")
            save_pdf(resp, save_path)
        else:
            print(f"  PDF already exists")
        
        saved_files.append(save_path)
        return saved_files
    
    if "html" in resp.headers.get("Content-Type", "").lower():
        print(f"  Parsing HTML index...")
        pdf_links = extract_pdf_links(resp.text, base_url)
        print(f"  Found {len(pdf_links)} PDF links")
        
        for link in pdf_links:
            if use_delay:
                time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
            
            pdf_resp = safe_get(link, stream=True)
            if pdf_resp and is_pdf_response(pdf_resp):
                filename = os.path.basename(link)
                save_path = os.path.join(target_dir, filename)
                
                if not os.path.exists(save_path):
                    print(f"    Downloading {filename}")
                    save_pdf(pdf_resp, save_path)
                
                saved_files.append(save_path)
    
    return saved_files


### 6. Strategy Selection

In [ ]:
def scrape_session_year(session, year, use_delay=True):
    """Attempt modern index-based download first"""
    print(f"\n=== Session {session} Year {year} ===")

    files = download_from_index(session, year, use_delay)
    
    if len(files) > 0:
        print(f"✓ Downloaded {len(files)} files via index")
        return files
    
    print(f"✗ No files found for {year}")
    return []

### 7. Main Execution

In [ ]:
def scrape_all(use_delay=True):
    """
    Main orchestrator function
    Logic:
    1. Try modern method for each session/year pair
    2. If modern method fails for entire session, fall back to legacy method
    3. Track processed sessions to avoid duplicate attempts
    """
    all_files = []
    processed_sessions = set()  
    
    for session, years in SESSION_YEARS.items():
        new_version_success = False
        for year in years:
            try:
                files = scrape_session_year(session, year, use_delay)
                if len(files) > 0:
                    all_files.extend(files)
                    new_version_success = True
            except Exception as e:
                print(f"✗ Error: {e}")
        
        if not new_version_success and session not in processed_sessions:
            print(f"\n=== Session {session} ({'-'.join(map(str, years))}) - Using Old Method ===")
            try:
                files = download_by_rollcall_number(session, use_delay)
                all_files.extend(files)
                processed_sessions.add(session) 
                print(f"✓ Downloaded {len(files)} files via Roll Call traversal")
            except Exception as e:
                print(f"✗ Error: {e}")
    
    print(f"\n{'='*50}")
    print(f"Total files downloaded: {len(all_files)}")
    print(f"{'='*50}")
    
    return all_files

In [ ]:
if __name__ == "__main__":
    files = scrape_all(use_delay=True)


=== Session 182 Year 2001 ===
  Trying index page: https://malegislature.gov/Journal/House/182/2001/RollCalls
✗ No files found for 2001

=== Session 182 Year 2002 ===
  Trying index page: https://malegislature.gov/Journal/House/182/2002/RollCalls
✗ No files found for 2002

=== Session 182 (2001-2002) - Using Old Method ===
    #1 not found
  Stopped at Roll Call #5 (consecutive fails: 5)
✓ Downloaded 0 files via Roll Call traversal

=== Session 183 Year 2003 ===
  Trying index page: https://malegislature.gov/Journal/House/183/2003/RollCalls
✗ No files found for 2003

=== Session 183 Year 2004 ===
  Trying index page: https://malegislature.gov/Journal/House/183/2004/RollCalls
✗ No files found for 2004

=== Session 183 (2003-2004) - Using Old Method ===
    #1 downloading...
    #2 downloading...
    #3 downloading...
    #4 downloading...
    #5 downloading...
    #6 downloading...
    #7 downloading...
    #8 downloading...
    #9 downloading...
    #10 downloading...
    #11 download